# SC2001 Project 1 - Part (c): Analyze Time Complexity

Algorithms (part (a)) are imported unchanged from `SC2001 Lab Section abc.i.py`.
The analysis is redefined here, fixing five problems in the `.py` version:

| # | Problem | Fix |
|---|---------|-----|
| 1 | (c.iii) chose S by key comparisons, which always favours a tiny S where the hybrid is just merge sort | choose by CPU time |
| 2 | (c.iii) grid `2, 5, 10, 20...` skipped the real minimum at S=3 | every small integer |
| 3 | (c.ii) too coarse, started past the minimum, hid the staircase | S = 1..32, step plot + zoom |
| 4 | (c.i) log-log can't distinguish O(n) from O(n log n) | added `comparisons / (n log2 n)` panel |
| 5 | no seed, datasets regenerated per part | seeded per size, freed after use |

## Setup

In [ ]:
%matplotlib inline

import gc
import importlib.util
import math
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt

# The source file name contains spaces and a dot, so it cannot be imported with
# a plain `import` statement. Load it by path instead.
MODULE_PATH = Path("SC2001 Lab Section abc.i.py").resolve()
if not MODULE_PATH.exists():
    raise FileNotFoundError(f"Could not find {MODULE_PATH}. Run this notebook from the same folder as the .py file.")

spec = importlib.util.spec_from_file_location("hybrid_sort_lib", MODULE_PATH)
lib = importlib.util.module_from_spec(spec)
spec.loader.exec_module(lib)   # the .py has an `if __name__ == "__main__"` guard, so nothing runs on import

# Part (a): the algorithms, imported unchanged.
insertion_sort = lib.insertion_sort
merge          = lib.merge
hybrid_sort    = lib.hybrid_sort

print("Loaded algorithms from:", MODULE_PATH.name)

## Parameters

`QUICK = True` shrinks everything to under a minute. `False` is the report run:
~8 minutes, peaking near 1 GB at n = 10,000,000.

In [ ]:
QUICK = False          # True = fast development run, False = full run for the report

SEED = 2001            # fixes the datasets so the numbers in the report are reproducible
MAX_VALUE_X = 1000000  # datasets hold random integers in [1, MAX_VALUE_X]

FIXED_S = 10           # the fixed threshold S used in part (c.i)
FIXED_N = 1000000      # the fixed input size n used in part (c.ii)

# In (c.iii), any S whose CPU time is within this fraction of the best is
# treated as tied with it. Timing on a desktop is not precise enough to crown a
# single winner, so we report the near-optimal band instead.
PLATEAU_TOL = 0.05

if QUICK:
    SIZES_CI      = [1000, 5000, 10000, 50000, 100000]
    FIXED_N       = 100000
    S_VALUES_CII  = list(range(1, 17)) + [24, 32, 48, 64]
    SIZES_CIII    = [1000, 10000, 100000]
    S_VALUES_CIII = [1, 2, 3, 4, 6, 8, 12, 16, 24, 32, 48, 64]
else:
    # (c.i) 13 sizes from 1,000 to 10,000,000, roughly evenly spaced on a log scale
    SIZES_CI = [1000, 2500, 5000, 10000, 25000, 50000, 100000,
                250000, 500000, 1000000, 2500000, 5000000, 10000000]
    # (c.ii) every integer through the interesting region, then a coarse tail
    S_VALUES_CII = list(range(1, 33)) + [40, 48, 64, 96, 128]
    # (c.iii) four input sizes, moderate S grid resolved at the low end
    SIZES_CIII    = [1000, 10000, 100000, 1000000]
    S_VALUES_CIII = [1, 2, 3, 4, 6, 8, 10, 12, 16, 20, 24, 32, 40, 48, 64, 96, 128]

print(f"QUICK = {QUICK}")
print(f"(c.i)   sizes: {SIZES_CI}")
print(f"(c.ii)  n = {FIXED_N:,}, {len(S_VALUES_CII)} values of S from {min(S_VALUES_CII)} to {max(S_VALUES_CII)}")
print(f"(c.iii) sizes: {SIZES_CIII}, {len(S_VALUES_CIII)} values of S")

## Part (b): input data

`generate_one` seeds from `n`, so a given size yields the identical array in
every part. Arrays are built on demand and released after use rather than all
held at once.

In [ ]:
def generate_one(n, x=MAX_VALUE_X, seed=SEED):
    """Generates one array of n random integers in [1, x], reproducibly.

    The seed is derived from n, so calling this twice with the same n always
    returns the same array, while different sizes still get independent data.
    """
    random.seed(seed * 1000003 + n)
    return [random.randint(1, x) for _ in range(n)]


def time_sort(sort_fn, data, min_total=0.5, min_repeats=2, max_repeats=200):
    """Times sort_fn(array); returns (best_seconds, comparisons, runs).

    Timing a sort on a busy desktop is noisy enough to invent fake winners, so
    this follows what `timeit` does:

      * REPEAT ON A TIME BUDGET, not a fixed count. Keep running until either
        `min_total` seconds of work have accumulated or `max_repeats` is hit.
        A 1.4 ms sort is therefore measured hundreds of times while a 4 s sort
        is measured twice - each gets about the same total attention, and cheap
        inputs (the noisiest ones) get the most samples.
      * KEEP THE MINIMUM, not the mean. The fastest run is the one least
        disturbed by other activity; slow runs record what else the OS was
        doing, so averaging them in imports that noise instead of cancelling it.
      * DISABLE THE GARBAGE COLLECTOR. Merge sort allocates a new list at every
        node, so a collection cycle can fire mid-sort and land entirely on one
        S value. Reference counting still reclaims the temporaries, so nothing
        leaks; only the cyclic collector pauses.
      * COPY OUTSIDE THE TIMED REGION, so we measure the sort and not the copy.
    """
    best = float("inf")
    comparisons = None
    total = 0.0
    runs = 0
    gc_was_enabled = gc.isenabled()
    gc.disable()
    try:
        while runs < max_repeats and (runs < min_repeats or total < min_total):
            working = data[:]                      # not timed
            start = time.perf_counter()
            _, comparisons = sort_fn(working)
            elapsed = time.perf_counter() - start
            best = min(best, elapsed)
            total += elapsed
            runs += 1
    finally:
        if gc_was_enabled:
            gc.enable()
    return best, comparisons, runs


# Sanity checks: the algorithm sorts correctly and the generator is reproducible
_probe = generate_one(2000, 1000)
assert hybrid_sort(_probe, 1)[0] == sorted(_probe)     # S=1 is exactly plain merge sort
assert hybrid_sort(_probe, 16)[0] == sorted(_probe)
assert generate_one(2000, 1000) == _probe, "generate_one is not reproducible"
print("Sanity checks passed.")
del _probe

---
## Part (c.i): key comparisons vs n, with S fixed

Left panel is the required plot. On log-log axes it cannot test the theory -
n and n log n have slopes 1.00 and ~1.03, so any smooth growth looks straight.
The right panel divides the theory out: `comparisons / (n log2 n)` is constant
only if the algorithm really is Theta(n log n), and sags if it is Theta(n). Its
y-axis starts at 0 so the flatness is not a zoom artefact.

In [ ]:
def analyze_comparisons_vs_n(sizes, fixed_s, verbose=True):
    """Part (c.i). Datasets are generated one at a time and released straight
    after sorting, so peak memory is roughly one array rather than all of them."""
    measured, comparisons, seconds = [], [], []

    for n in sizes:
        if verbose:
            print(f"n = {n:>10,}  generating...", end="", flush=True)
        data = generate_one(n)
        if verbose:
            print(" sorting...", end="", flush=True)
        start = time.perf_counter()
        _, comp = hybrid_sort(data, fixed_s)
        elapsed = time.perf_counter() - start
        data = None                     # release before building the next, larger array
        measured.append(n)
        comparisons.append(comp)
        seconds.append(elapsed)
        if verbose:
            print(f" {comp:>13,} comparisons  ({elapsed:6.2f}s)")

    ratios = [c / (n * math.log2(n)) for n, c in zip(measured, comparisons)]
    c_fit = sum(ratios) / len(ratios)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

    ax1.plot(measured, comparisons, marker="o", color="tab:blue",
             label=f"Empirical comparisons (S={fixed_s})")
    ax1.plot(measured, [c_fit * n * math.log2(n) for n in measured],
             linestyle="--", color="tab:red", label=f"{c_fit:.3f} x n log2(n)")
    ax1.set_xscale("log")
    ax1.set_yscale("log")
    ax1.set_title(f"Key comparisons vs input size (S = {fixed_s})")
    ax1.set_xlabel("Input size n")
    ax1.set_ylabel("Key comparisons")
    ax1.grid(True, which="both", ls="--", lw=0.5)
    ax1.legend()

    ax2.plot(measured, ratios, marker="o", color="tab:blue",
             label="comparisons / (n log2 n)")
    ax2.axhline(c_fit, linestyle="--", color="tab:red", label=f"mean = {c_fit:.4f}")
    ax2.set_xscale("log")
    ax2.set_ylim(0, max(ratios) * 1.3)     # keep 0 in view: no fake flatness
    ax2.set_title("Theory test: a flat line means Theta(n log n)")
    ax2.set_xlabel("Input size n")
    ax2.set_ylabel("comparisons / (n log2 n)")
    ax2.grid(True, which="both", ls="--", lw=0.5)
    ax2.legend()

    fig.tight_layout()
    plt.show()

    spread = (max(ratios) - min(ratios)) / c_fit * 100
    decades = math.log10(max(measured) / min(measured))
    print(f"\nRatio comparisons/(n log2 n): mean {c_fit:.4f}, "
          f"range {min(ratios):.4f} to {max(ratios):.4f} ({spread:.1f}% spread)")
    print(f"Ratio holds within {spread:.1f}% across {decades:.0f} orders of magnitude of n: "
          f"the evidence for Theta(n log n).")
    return measured, comparisons, ratios, seconds

In [ ]:
sizes_ci, comps_ci, ratios_ci, times_ci = analyze_comparisons_vs_n(SIZES_CI, FIXED_S)

---
## Part (c.ii): key comparisons vs S, with n fixed

**The curve is a staircase.** Merge sort halves the array regardless of S, so the
only subarray sizes it can hand to insertion sort are `n / 2^k` - for
n = 1,000,000 those are `... 122, 61, 30, 15, 7 ...`. S only picks which rung you
stop at, so S=75 and S=100 both stop at 61 and give identical counts. Hence
`steps-post` rather than straight lines between samples.

The right panel zooms on small S where the steps show. The rise is insertion
sort's Theta(S^2) per block overtaking the Theta(n log S) of merge comparisons
it saves.

In [ ]:
def analyze_comparisons_vs_s(n, s_values, zoom_max=16, verbose=True):
    """Part (c.ii)."""
    data = generate_one(n)
    s_list, comparisons = [], []

    for s in s_values:
        start = time.perf_counter()
        _, comp = hybrid_sort(data, s)
        elapsed = time.perf_counter() - start
        s_list.append(s)
        comparisons.append(comp)
        if verbose:
            print(f"S = {s:>4}   {comp:>13,} comparisons  ({elapsed:5.2f}s)")

    best_i = min(range(len(s_list)), key=lambda i: (comparisons[i], s_list[i]))
    best_s, best_comp = s_list[best_i], comparisons[best_i]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

    ax1.step(s_list, comparisons, where="post", color="tab:green")
    ax1.plot(s_list, comparisons, "o", ms=4, color="tab:green")
    ax1.set_title(f"Key comparisons vs S (n = {n:,})")
    ax1.set_xlabel("Threshold S")
    ax1.set_ylabel("Key comparisons")
    ax1.grid(True, ls="--", lw=0.5)

    zoom = [(s, c) for s, c in zip(s_list, comparisons) if s <= zoom_max]
    zs = [s for s, _ in zoom]
    zc = [c for _, c in zoom]
    ax2.step(zs, zc, where="post", color="tab:green")
    ax2.plot(zs, zc, "o", ms=5, color="tab:green")
    ax2.axvline(best_s, ls=":", color="tab:red", label=f"minimum at S = {best_s}")
    ax2.set_title(f"Zoom on S <= {zoom_max}: the staircase")
    ax2.set_xlabel("Threshold S")
    ax2.set_ylabel("Key comparisons")
    ax2.grid(True, ls="--", lw=0.5)
    ax2.legend()

    fig.tight_layout()
    plt.show()

    # The rungs the recursion can actually stop on, which is what explains the flats
    rungs, size = [], n
    while size > 1:
        size /= 2
        if size <= max(s_list) * 2:
            rungs.append(round(size, 1))
        if size < 1:
            break
    worst = max(comparisons)
    print(f"\nRungs the recursion can reach (n / 2^k): {rungs[:9]}")
    print(f"Fewest comparisons: S = {best_s} ({best_comp:,})")
    print(f"Most:               S = {s_list[comparisons.index(worst)]} ({worst:,}), "
          f"{worst / best_comp:.2f}x the minimum")
    return s_list, comparisons, best_s

In [ ]:
s_list_cii, comps_cii, best_s_by_comparisons = analyze_comparisons_vs_s(FIXED_N, S_VALUES_CII)

---
## Part (c.iii): determining the optimal S

The comparison-optimal S is ~3, and it beats its neighbours by 0.0004%. At that
S the insertion sort does nothing, so the "optimal hybrid" is just merge sort
again - key comparisons do not measure what the hybrid is *for*. What merge sort
pays at small subarrays is **overhead**: every recursive call is a function call
plus two fresh list allocations, invisible to a comparison counter but plain on
a clock. So S is chosen on **CPU time**, with both criteria reported.

Timings are not reproducible, so two safeguards. `time_sort` repeats on a budget
with GC off and keeps the fastest run. And the optimum comes from the
size-averaged curve (thick black), not any one size's argmin - averaging four
sizes suppresses the jitter. The shaded band is every S within `PLATEAU_TOL` of
that minimum; the valley is broad, so the band, not a single integer, is the
answer.

Each size's curve is normalised by its own best time so all four share one axis.
Valleys that line up mean the optimal S is independent of n, as theory predicts:
S balances two constant factors, not anything that grows with n.

In [ ]:
def find_optimal_s(sizes, s_values, verbose=True):
    """Part (c.iii). Selects S by CPU time; also records key comparisons."""
    results = {}

    for n in sizes:
        data = generate_one(n)
        if verbose:
            print(f"\nn = {n:,}")
        times, comparisons = [], []
        for s in s_values:
            secs, comp, runs = time_sort(lambda a, s=s: hybrid_sort(a, s), data)
            times.append(secs)
            comparisons.append(comp)
            if verbose:
                print(f"   S = {s:>4}   {secs * 1000:9.2f} ms  (best of {runs:>3})"
                      f"   {comp:>13,} comparisons")
        data = None

        i_time = min(range(len(s_values)), key=lambda i: times[i])
        i_comp = min(range(len(s_values)), key=lambda i: (comparisons[i], s_values[i]))
        best_time = times[i_time]
        # Any S whose time is within tolerance of the best is statistically
        # indistinguishable from it. Reporting that band is more honest than
        # reporting a single argmin that timing noise could have picked.
        plateau = [s for s, t in zip(s_values, times) if t <= best_time * (1 + PLATEAU_TOL)]
        results[n] = {
            "s_values": list(s_values),
            "times": times,
            "comparisons": comparisons,
            "best_s_time": s_values[i_time],
            "best_time": best_time,
            "plateau": plateau,
            "best_s_comparisons": s_values[i_comp],
            "best_comparisons": comparisons[i_comp],
            "time_at_s1": times[s_values.index(1)] if 1 in s_values else None,
        }
        if verbose:
            r = results[n]
            print(f"   -> fastest: S = {r['best_s_time']} ({best_time * 1000:.1f} ms);"
                  f" within {PLATEAU_TOL:.0%}: S in {plateau[0]}..{plateau[-1]};"
                  f" fewest comparisons: S = {r['best_s_comparisons']}")

    # Aggregate across input sizes. For each S, express that size's time as a
    # multiple of the best time achieved at the same size, then average over
    # sizes. This is the robust way to answer "using different sizes of input
    # datasets, determine an optimal S": one size's argmin sits within noise of
    # its neighbours, but a curve averaged over four sizes does not.
    aggregate = []
    for i, s in enumerate(s_values):
        aggregate.append(sum(r["times"][i] / min(r["times"]) for r in results.values())
                         / len(results))
    i_best = min(range(len(s_values)), key=lambda i: aggregate[i])
    best_s_overall = s_values[i_best]
    band = [s for s, a in zip(s_values, aggregate)
            if a <= aggregate[i_best] * (1 + PLATEAU_TOL)]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5.5))

    if band:
        ax1.axvspan(min(band), max(band), color="tab:green", alpha=0.12,
                    label=f"within {PLATEAU_TOL:.0%} of the aggregate best")
    for n, r in results.items():
        floor = min(r["times"])
        ax1.plot(r["s_values"], [t / floor for t in r["times"]],
                 marker="o", ms=3, lw=1, alpha=0.65, label=f"n = {n:,}")
    ax1.plot(s_values, aggregate, color="black", lw=2.2, marker="o", ms=4,
             label="mean over all sizes", zorder=5)
    ax1.axvline(best_s_overall, ls=":", color="black", lw=1,
                label=f"optimal S = {best_s_overall}")
    ax1.axhline(1.0, ls="--", lw=0.8, color="grey")
    ax1.set_xscale("log", base=2)
    ax1.set_xticks(s_values)
    ax1.set_xticklabels(s_values, fontsize=7)
    ax1.set_title("Normalised CPU time vs S (1.0 = each size's own best)")
    ax1.set_xlabel("Threshold S")
    ax1.set_ylabel("Time / best time for that n")
    ax1.grid(True, which="both", ls="--", lw=0.5)
    ax1.legend()

    ns = list(results.keys())
    ax2.plot(ns, [results[n]["best_s_time"] for n in ns],
             marker="o", color="tab:purple", label="Optimal S (by CPU time)")
    ax2.plot(ns, [results[n]["best_s_comparisons"] for n in ns],
             marker="s", ls="--", color="tab:orange", label="Optimal S (by key comparisons)")
    ax2.set_xscale("log")
    ax2.set_title("Optimal threshold S vs input size n")
    ax2.set_xlabel("Input size n")
    ax2.set_ylabel("Optimal S")
    ax2.grid(True, which="both", ls="--", lw=0.5)
    ax2.legend()

    fig.tight_layout()
    plt.show()

    print("\n" + "=" * 92)
    print(f"{'n':>10}  {'best S':>7}  {'best time':>11}  {'vs S=1':>7}  "
          f"{'S within ' + format(PLATEAU_TOL, '.0%'):>16}  {'best S (comps)':>14}")
    print("-" * 92)
    for n, r in results.items():
        speedup = f"{r['time_at_s1'] / r['best_time']:.2f}x" if r["time_at_s1"] else "n/a"
        row_band = f"{r['plateau'][0]}..{r['plateau'][-1]}"
        print(f"{n:>10,}  {r['best_s_time']:>7}  {r['best_time'] * 1000:>8.1f} ms  "
              f"{speedup:>7}  {row_band:>16}  {r['best_s_comparisons']:>14}")
    print("=" * 92)
    print(f"Aggregated over all sizes: optimal S = {best_s_overall}; "
          f"S in {min(band)}..{max(band)} is within {PLATEAU_TOL:.0%} of it.")
    return {"per_n": results, "s_values": list(s_values), "aggregate": aggregate,
            "best_s_overall": best_s_overall, "band": band}

In [ ]:
results_ciii = find_optimal_s(SIZES_CIII, S_VALUES_CIII)

---
## Conclusion, and the value carried into part (d)

`OPTIMAL_S` is the minimum of the size-averaged CPU-time curve from (c.iii).

In [ ]:
per_n = results_ciii["per_n"]
band  = results_ciii["band"]

# Taken from the curve aggregated over all input sizes, not from any single
# size's argmin, which timing noise can move by several steps.
OPTIMAL_S = results_ciii["best_s_overall"]

by_time = [per_n[n]["best_s_time"] for n in per_n]
by_comp = [per_n[n]["best_s_comparisons"] for n in per_n]

print(f"Optimal S by CPU time, per input size:        {by_time}")
print(f"Optimal S by key comparisons, per input size: {by_comp}")
print(f"Within {PLATEAU_TOL:.0%} of the aggregate best:            S in {min(band)}..{max(band)}")
print()
print(f"OPTIMAL_S = {OPTIMAL_S}   (minimum of the size-averaged timing curve)")
print()
print(" (c.i)   ratio flat in n  ->  Theta(n log n) for fixed S, as predicted.")
print(" (c.ii)  comparisons rise with S in a staircase: insertion sort's Theta(S^2) per")
print("         block overtakes the Theta(n log S) of merge comparisons it saves.")
print(" (c.iii) comparison-optimal S is ~3 and worth 0.0004%; time-optimal S is far larger")
print("         and worth a real speedup. Comparisons cannot see recursion and allocation")
print("         overhead, which is what the hybrid actually saves.")